In [ ]:
! pip install --upgrade -q pandas numpy scikit-learn imbalanced-learn optuna plotly nbformat

In [ ]:
from collections import Counter
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_sample_weight

import xgboost as xgb
from xgboost import XGBClassifier

import cupy as cp

import optuna

from utils.constants import *

In [ ]:
df = pd.read_csv("../data/3_gold/dataset-processed-gb.csv")

X = df.drop("class", axis=1)
y = df["class"]
y = y.map(TARGET_LABEL_MAP)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=TEST_RATIO, random_state=RANDOM_STATE, stratify=y
)

In [ ]:
def get_coarse_fine_data(X, y):
    X_coarse = X.copy()
    y_coarse = y.copy()
    y_coarse = y_coarse.map(COARSE_LABEL_MAP)

    high_risk_mask = y.isin([1, 2])
    X_fine = X[high_risk_mask].copy()
    y_fine = y[high_risk_mask].copy()
    y_fine = y_fine.map(FINE_LABEL_MAP)

    return X_coarse, y_coarse, X_fine, y_fine

In [ ]:
def predict_soft_cascade(model_coarse, model_fine, X, y):
    # Get Probabilities from Coarse
    probs_coarse = model_coarse.predict_proba(X)
    p_high_risk = probs_coarse[:, 1]
    
    # Get Probabilities from Fine
    probs_fine = model_fine.predict_proba(X) 
    p_severe_given_high = probs_fine[:, 1] # P(Severe | High)
        
    # P(Severe) = P(High) * P(Severe | High)
    p_severe_global = p_high_risk * p_severe_given_high
    
    # P(Alarm) = P(High) * (1 - P(Severe | High))
    p_alarm_global = p_high_risk * (1 - p_severe_given_high)
    
    # P(Low) = 1 - P(High)
    p_low_global = 1.0 - p_high_risk
    
    final_probs = np.vstack([p_low_global, p_alarm_global, p_severe_global]).T
    final_preds = np.argmax(final_probs, axis=1)

    return f1_score(y, final_preds, average='macro'), final_preds


def predict_hard_cascade(
    model_coarse, model_fine, X, y, 
    threshold_coarse=0.5, threshold_fine=0.5
):
    # Predict Coarse (0 = Low, 1 = High)
    probs_coarse = model_coarse.predict_proba(X)[:, 1]
    
    preds_coarse = (probs_coarse >= threshold_coarse).astype(int)
    final_preds = preds_coarse.copy()
    
    high_risk_indices = np.where(preds_coarse == 1)[0]
    
    if len(high_risk_indices) > 0:
        X_high_risk = X.iloc[high_risk_indices]
        
        # Predict Fine (0 = Alarm, 1 = Severe)
        probs_fine_local = model_fine.predict_proba(X_high_risk)[:, 1]
        preds_fine_local = (probs_fine_local >= threshold_fine).astype(int)
        
        # Map Fine predictions back to Global labels
        preds_fine_global = np.array([FINE_LABEL_MAP_REVERSE[p] for p in preds_fine_local])
        final_preds[high_risk_indices] = preds_fine_global

    return f1_score(y, final_preds, average='macro'), final_preds

In [ ]:
FIXED_PARAMS = {
    "n_estimators": 3000,
    "early_stopping_rounds": 10,
    "enable_categorical": True,
    "objective": "multi:softprob",
    "num_class": N_CLASSES,
    "tree_method": "hist",
    "device": "cuda",
    "eval_metric": "mlogloss",
    "random_state": RANDOM_STATE,
    "verbosity": 0
}

In [ ]:
def train_on_folds(params_coarse, params_fine, use_soft_cascade, threshold_coarse=None, threshold_fine=None, gpu=True):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    scores = []

    for train_idx, valid_idx in skf.split(X_train_full, y_train_full):
        X_train, y_train = X_train_full.iloc[train_idx], y_train_full.iloc[train_idx]
        X_valid, y_valid = X_train_full.iloc[valid_idx], y_train_full.iloc[valid_idx]

        sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

        if gpu:
            X_train, y_train = cp.asarray(X_train), cp.asarray(y_train)
            X_valid, y_valid = cp.asarray(X_valid), cp.asarray(y_valid)

        clf_coarse = XGBClassifier(**params_coarse)
        clf_fine = XGBClassifier(**params_fine)

        X_coarse, y_coarse, X_fine, y_fine = get_coarse_fine_data(X_train, y_train)

        X_coarse_valid, y_coarse_valid, X_fine_valid, y_fine_valid = get_coarse_fine_data(X_valid, y_valid)

        try:
            clf_coarse.fit(
                X_coarse, y_coarse,
                eval_set=[(X_coarse_valid, y_coarse_valid)], 
                sample_weight=sample_weights, 
                verbose=False
            )
            clf_fine.fit(
                X_fine, y_fine,
                eval_set=[(X_fine_valid, y_fine_valid)], 
                sample_weight=sample_weights, 
                verbose=False
            )

            if use_soft_cascade:
                f1, _ = predict_soft_cascade(clf_coarse, clf_fine, X_valid, y_valid)
            else:
                f1, _ = predict_hard_cascade(
                    clf_coarse, clf_fine, X_valid, y_valid,
                    threshold_coarse, threshold_fine
                )
            
            scores.append(f1)
        except ValueError:
            return 0.0, 1.0
        
    return np.mean(scores), np.std(scores)


def objective(trial: optuna.trial.Trial):
    params_coarse = {
        "max_depth": trial.suggest_int("max_depth_coarse", 2, 15),
        "learning_rate": trial.suggest_float("learning_rate_coarse", 1e-3, 1e-1, step=0.01),
        "subsample": trial.suggest_float("subsample_coarse", 0.5, 1.0, step=0.1),
        "colsample_bytree": trial.suggest_float("colsample_bytree_coarse", 0.1, 1.0, step=0.1),
        "reg_alpha": trial.suggest_int("reg_alpha_coarse", 1, 50),
        "reg_lambda": trial.suggest_int("reg_lambda_coarse", 1, 50),
        "min_child_weight": trial.suggest_int("min_child_weight_coarse", 1, 10),
        "gamma": trial.suggest_float("gamma_coarse", 0, 5, step=0.5),
        **FIXED_PARAMS
    }
    params_fine = {
        "max_depth": trial.suggest_int("max_depth_fine", 2, 15),
        "learning_rate": trial.suggest_float("learning_rate_fine", 1e-3, 1e-1, step=0.01),
        "subsample": trial.suggest_float("subsample_fine", 0.5, 1.0, step=0.1),
        "colsample_bytree": trial.suggest_float("colsample_bytree_fine", 0.1, 1.0, step=0.1),
        "reg_alpha": trial.suggest_int("reg_alpha_fine", 1, 50),
        "reg_lambda": trial.suggest_int("reg_lambda_fine", 1, 50),
        "min_child_weight": trial.suggest_int("min_child_weight_fine", 1, 10),
        "gamma": trial.suggest_float("gamma_fine_fine", 0, 5, step=0.5),
        **FIXED_PARAMS
    }

    mode = trial.suggest_categorical("mode", choices=["soft", "hard"])

    threshold_coarse = None
    threshold_fine = None
    use_soft_cascade = (mode == "hard")
    
    if not use_soft_cascade:
        threshold_coarse = trial.suggest_float("threshold_coarse", low=0.25, high=0.55)
        threshold_fine = trial.suggest_float("threshold_fine", low=0.25, high=0.55)

    avg, _ = train_on_folds(params_coarse, params_fine, use_soft_cascade, threshold_coarse, threshold_fine, gpu=True)
    return avg

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, timeout=14400, show_progress_bar=True, n_jobs=-1)

best_trial = study.best_trial
print('Best F1:', best_trial.value)
print('Best Params:', best_trial.params)

In [ ]:
import pickle
from pathlib import Path

output_dir = Path('../results/optimization/random_forest')
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / 'optuna_study.pkl', 'wb') as fout:
    pickle.dump(study.sampler, fout)

In [ ]:
import optuna.visualization as vis

display(vis.plot_param_importances(study))
display(vis.plot_optimization_history(study))

In [ ]:
avg_f1_final, std_f1_final = train_on_folds(best_trial.params)